# 01 · Capacity — how many units does each region need?

*Task 01 of the rebuilt mRNN analysis. Task 00 settled how to train; this is the first of
the structural questions.*

**The framing.** With roughly twenty thousand free parameters against fifty thousand
target numbers, plenty of configurations fit these trajectories. Ranking them by loss is
therefore not very interesting. The useful question is the opposite one: **which
constraints can the data tolerate?** A model that still reproduces the trajectories while
being restricted tells us something; a model that fits because it was given enough
freedom to fit anything does not.

This notebook applies the first constraint — network width — and asks how far it can be
pushed before the fit gives way.

**Why the range stops at 60.** Each region reads out to 42 principal components, so a
region with many more than 42 units is not being asked for anything its width could
supply. The sweep covers **20, 30, 40, 50, 60 units per region**, which brackets the
target dimensionality from well below to somewhat above.

**Why there is no bottleneck here.** The task-00 recipe carried
`recurrent_bottleneck_dim = 3`, a rank constraint on every inter-regional block that was
inherited rather than tested. That is itself one of the constraints this chapter is about,
so it does not belong in the baseline. Every model here uses **dense, full-rank
inter-regional connectivity**, and the rank constraint is imposed and measured in task 03.

> Expressing that required a code change: inter-region blocks were *always* factorized as
> `left @ right`, so a model with no rank constraint could not be built. Setting the rank
> equal to the hidden width is not a substitute — the product parameterization carries
> twice the parameters and optimizes differently.

**And no within-region L1 either.** The first pass inherited `l1_weight_scale = 0.01` from
the legacy runs. Measured on a fitted model that is not a mild sparsity prior: it drives
the within-region blocks to ~1e-6 against ~1e-1 for the cross-region blocks — five orders
of magnitude — **with no change in fit**. So it was silently choosing one of two equally
good solutions, and removing a whole class of connection from every model in the project.
Sparsity is swept properly in task 04; the baseline here has none.

| Section | |
|---|---|
| 1 | The sweep, and what it inherits |
| 2 | Run state and submission (off by default) |
| 3 | Loss trajectories and convergence |
| 4 | Fit against the noise ceiling, and the parameters that bought it |
| 5 | The visual check, across the whole sweep |
| 6 | Do the seeds agree? |
| 7 | Selection |

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, Markdown, display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from dal_monte_2022_analysis.config.load import load_config
from dal_monte_2022_analysis.ephys.analysis import fixation_mrnn_protocol as protocol
from dal_monte_2022_analysis.ephys.analysis import fixation_mrnn_sweep as sweep
from dal_monte_2022_analysis.ephys.analysis import fixation_mrnn_synthesis as syn
from dal_monte_2022_analysis.ephys.analysis import fixation_mrnn_target_loss as tl
from dal_monte_2022_analysis.ephys.analysis import fixation_psth_noise_ceiling as ceiling_mod
from dal_monte_2022_analysis.ephys.plotting import fixation_mrnn_sweep as viz
from dal_monte_2022_analysis.ephys.plotting.thesis_common import (
    ThesisFigureSettings,
    apply_thesis_plot_style,
    figure_to_png_bytes,
    save_thesis_figure,
)

DATASET_CFG_PATH = repo_root / "configs" / "dataset.yaml"
MRNN_CFG_PATH = repo_root / "configs" / "ephys_fixation_mrnn.yaml"
apply_thesis_plot_style(load_config(repo_root / "configs" / "plotting.yaml"))

PROTOCOL_ROOT = protocol.resolve_chapter_root(DATASET_CFG_PATH, task="00_training_protocol")
TASK_ROOT = sweep.resolve_task_root("01_capacity", DATASET_CFG_PATH)
CEILING_DIR = ceiling_mod.resolve_output_dir(DATASET_CFG_PATH)
FIGURE_DIR = syn.resolve_output_dir(DATASET_CFG_PATH, scope="01_capacity")
FIGURES = ThesisFigureSettings(output_dir=FIGURE_DIR)

SELECTED_PROTOCOL = sweep.load_selected_protocol(PROTOCOL_ROOT / "selected_protocol.yaml")

#: Widths to test. Each region reads out 42 PCs, so far above that adds freedom the
#: target cannot use.
HIDDEN_UNIT_GRID = (20, 30, 40, 50, 60)
#: Seeds per cell. Three is a screen; seed agreement is measured over the three pairs it
#: gives, which is thin but enough to rank variants. Raise it for the model that survives.
SWEEP_SEEDS = 3
#: Region whose traces the gallery shows. Fixed across the sweep so rows compare.
GALLERY_REGION = "ofc"


def show(figure, stem: str) -> None:
    save_thesis_figure(figure, FIGURES, stem)
    display(Image(data=figure_to_png_bytes(figure, dpi=190)))


print("inherited recipe :", SELECTED_PROTOCOL["selected_label"])
print("task root        :", TASK_ROOT)

## 1. The sweep

Everything except width and the rank constraint is inherited from the recipe task 00
froze, so exactly one thing differs between cells.

In [ ]:
# Both structural settings are stated explicitly rather than inherited. The rank
# constraint is what task 03 tests, and l1_weight_scale = 0.01 -- which the first pass
# inherited from the legacy runs -- is not a sparsity prior but an ablation: it drives the
# within-region blocks to ~1e-6 against ~1e-1 for the cross-region blocks with no change
# in fit. Task 04 sweeps it properly. Stating both here means this task does not depend on
# task 00 having been re-run first.
variants = [
    sweep.ModelVariant(
        label=f"h{units:02d}",
        overrides={
            "hidden_units": int(units),
            "recurrent_bottleneck_dim": None,
            "l1_weight_scale": 0.0,
        },
        arm="capacity",
    )
    for units in HIDDEN_UNIT_GRID
]
seeds = protocol.protocol_seeds(n_seeds=SWEEP_SEEDS)

display(pd.DataFrame([v.describe() for v in variants]))
display(pd.Series({
    **{k: v for k, v in SELECTED_PROTOCOL["optimizer"].items()},
    "iterations": SELECTED_PROTOCOL["epochs"],
    "connectivity": "full (all regions to all regions)",
    "inter-regional rank": "unconstrained (dense)",
    "PCs per region": SELECTED_PROTOCOL["architecture"]["pca_n_components"],
}, name="inherited").to_frame())
display(Markdown(
    f"**{len(variants)} widths × {len(seeds)} seeds = {len(variants) * len(seeds)} runs** at "
    f"{SELECTED_PROTOCOL['epochs']:,} iterations, submitted as one SLURM array job."
))

## 2. Run state and submission

Submission happens **only** if you set `SUBMIT = True`, and is blocked while a previously
submitted array is still on the queue.

In [ ]:
SUBMIT = False   # <-- set to True to actually submit the missing cells

commands, run_dirs = sweep.variant_job_commands(
    variants, seeds,
    root=TASK_ROOT, repo_root=repo_root,
    protocol=SELECTED_PROTOCOL, mrnn_cfg_path=MRNN_CFG_PATH,
)
inventory = sweep.index_variant_runs(TASK_ROOT, variants, seeds)
job_state = protocol.running_job_state(TASK_ROOT / "_jobs")

display(Markdown(
    f"**{int(inventory['complete'].sum())} complete**, **{int(inventory['diverged'].sum())} diverged**, "
    f"**{int(inventory['pending'].sum())} not yet run** of {len(inventory)} cells."
))
if job_state["active"]:
    display(Markdown(
        f"⚠️ **Job array `{job_state['job_id']}` is still on the queue** "
        f"({', '.join(f'{n} {s.lower()}' for s, n in sorted(job_state['states'].items()))}). "
        f"Submission is blocked."
    ))
elif commands:
    display(Markdown(f"{len(commands)} run(s) would be submitted."))
    print("first command:\n")
    print(commands[0])

In [ ]:
if job_state["active"]:
    display(Markdown(f"Nothing submitted: job array `{job_state['job_id']}` is still running."))
elif SUBMIT and commands:
    from dal_monte_2022_analysis.runtime.hpc.jobs import submit_dsq_array_job, write_job_file

    jobs_dir = TASK_ROOT / "_jobs"
    jobs_dir.mkdir(parents=True, exist_ok=True)
    job_file = jobs_dir / "capacity.txt"
    write_job_file(job_file, commands)
    job_id = submit_dsq_array_job(
        job_file_path=job_file,
        sbatch_script_path=jobs_dir / "capacity.sh",
        log_dir=jobs_dir / "logs",
        job_name="mrnn_capacity",
        partition="psych_gpu",
        cpus_per_task=1,
        mem_per_cpu="12G",
        time_limit="06:00:00",
        gres="gpu:1",
    )
    (jobs_dir / "job_id.txt").write_text(str(job_id) + "\n")
    display(Markdown(f"Submitted **{len(commands)}** runs as job array **{job_id}**."))
elif commands:
    display(Markdown("`SUBMIT` is **False** — nothing was submitted."))
else:
    display(Markdown("Every cell is already trained; go on to Section 3."))

## 3. Loss trajectories and convergence

Read this before any fit comparison. A width that merely makes the model harder to
optimise looks the same in the final loss as a width the data genuinely cannot use, and
only the trajectory separates them. Panel titles are green where every seed cleared the
task-00 convergence bar and red where they did not.

In [ ]:
# Runs fitted before the within-region L1 was removed answer a different question, and
# would be pooled silently with the corrected ones if nobody checked.
_stale = [
    path.parent for path in TASK_ROOT.glob("*/seed=*/run_config.yaml")
    if (path.parent / "checkpoint_best.pth").exists()
    and float(yaml.safe_load(path.read_text()).get("l1_weight_scale", 0.0)) > 0
]
if _stale:
    display(Markdown(
        f"🔴 **{len(_stale)} run(s) on disk were fitted with `l1_weight_scale > 0`**, which ablates "
        f"the within-region blocks rather than merely regularising them. They answer a different "
        f"question and must be refitted before anything below is read. Delete them, or move them "
        f"aside, and re-submit."
    ))

histories = sweep.load_histories(inventory)
labels = [v.label for v in variants if v.label in histories]

if not histories:
    display(Markdown("No completed runs yet — this section fills in as the sweep lands."))
else:
    convergence = sweep.convergence_table(histories)
    display(convergence.round(5))
    show(viz.plot_sweep_loss_trajectories(histories, convergence=convergence), "fig01_loss_trajectories")

## 4. Fit, and what it cost in parameters

Fit is scored against the **measured noise ceiling** — the split-half reliability of each
region's PC trajectories — rather than against 1.0, because scoring against 1.0 asks the
model to reproduce sampling noise.

The parameter count sits beside it because the two have to be read together. A fit
obtained with a tenth of the data's degrees of freedom means considerably more than the
same fit obtained with all of them.

In [ ]:
pc_ceiling = pd.read_csv(CEILING_DIR / "pc_space_ceiling.csv")
ceiling_by_region = pc_ceiling.groupby("region")["reliability"].mean().to_dict()

if not histories:
    display(Markdown("Nothing to score yet."))
else:
    fit = sweep.score_variant_fit(inventory, ceiling_by_region)
    parameters = pd.DataFrame([
        {"label": label, **sweep.count_trainable_parameters(
            inventory[(inventory["label"] == label) & inventory["complete"]].iloc[0]["run_dir"])}
        for label in labels
    ])
    display(parameters[["label", "within_region", "inter_region", "readout", "initial_state",
                        "total", "target_numbers", "parameters_per_datum"]].round(3))
    display(fit.groupby(["label", "condition"])["r2_vs_ceiling"].mean().unstack().round(4))
    show(viz.plot_fit_versus_constraint(fit, parameters, order=labels,
                                        x_label="hidden units per region"), "fig02_fit_vs_capacity")

## 5. The visual check, across the whole sweep

Looking at every trace of every model is not feasible, and a fit that scores well can
still be visibly wrong — that is how the interactive-face smoothing was found. The
compromise is a gallery: one row per width, all showing the **same three components of
the same region**, sized for scanning rather than for reading detail.

The model selected at the end gets the full-detail trace plots; everything else gets a
row here.

In [ ]:
if histories:
    pc_traces = sweep.gallery_traces(inventory, region=GALLERY_REGION, space="pc", indices=(0, 1, 2))
    show(viz.plot_fit_gallery(pc_traces, order=labels,
                              title=f"{GALLERY_REGION.upper()} — top three PCs, observed against mRNN"),
         "fig03_gallery_pc")

In [ ]:
if histories:
    fr_traces = sweep.gallery_traces(inventory, region=GALLERY_REGION, space="fr", indices=(0, 80, 160))
    show(viz.plot_fit_gallery(fr_traces, order=labels,
                              title=f"{GALLERY_REGION.upper()} — three example units, backprojected firing rate"),
         "fig04_gallery_fr")

## 6. Do the seeds agree?

The reason to care about width beyond fit. Seeds of the task-00 model agree at 0.997 on
what the model outputs and only 0.52 on its latent drive geometry — so the trajectories
are determined by the data while the mechanism producing them is not. A narrower network
has fewer ways to produce the same output, so if any constraint is going to make the
solution more identifiable, this is where it should start to show.

In [ ]:
if histories:
    agreement = pd.concat([
        tl.inter_seed_agreement(list(block["run_dir"])).assign(label=label)
        for label, block in inventory[inventory["complete"].astype(bool)].groupby("label", sort=False)
    ], ignore_index=True)
    display(agreement.pivot_table(index="label", columns="feature", values="mean_agreement").round(4))
    show(viz.plot_seed_agreement_by_variant(agreement, order=labels), "fig05_seed_agreement")

## 7. Selection

The rule follows the framing: **the narrowest width that reaches the noise ceiling**,
not the width with the highest score.

That distinction matters here. A ceiling-relative score above 1.0 means the model is
reproducing more than the split-half reliability says is reproducible — it is fitting
sampling noise, and "higher is better" stops being true. So width is not a quantity to
maximise; it is a constraint, and the interesting answer is the tightest one the data
still tolerates.

Seed agreement is reported alongside but does not override: if a narrower model is
markedly more identifiable at a small cost in fit, that is a judgement worth making
explicitly rather than by formula.

In [ ]:
if not histories:
    display(Markdown("Selection is deferred until the sweep completes."))
else:
    summary = (
        fit.groupby("label")["r2_vs_ceiling"].agg(["mean", "min"])
        .join(parameters.set_index("label")[["total", "parameters_per_datum"]])
        .join(agreement[agreement["feature"] == "latent drive geometry"]
              .set_index("label")["mean_agreement"].rename("seed_agreement_geometry"))
        .join(convergence.set_index("label")[["n_converged", "n_seeds"]])
        .loc[labels]
    )
    display(summary.round(4))

    converged = summary[summary["n_converged"] == summary["n_seeds"]]
    if converged.empty:
        display(Markdown("**No width converged on every seed.** That is the result; report it."))
    else:
        # The target is the ceiling, not the maximum. A score above 1.0 means the model is
        # reproducing more than the split-half reliability says is reproducible -- that is
        # overfitting, so "higher is better" is the wrong rule above 1.0.
        TOLERANCE = 0.01
        adequate = converged[converged["mean"] >= 1.0 - TOLERANCE]
        over = converged[converged["mean"] > 1.0 + TOLERANCE]
        if adequate.empty:
            winner = converged["mean"].idxmax()
            verdict = (
                f"No width reaches the noise ceiling; `{winner}` comes closest at "
                f"{float(converged.loc[winner, 'mean']):.3f}. Read that as a statement about the "
                f"model class rather than about width."
            )
        else:
            winner = adequate.index[0]
            verdict = (
                f"**Selected: `{winner}`** — the narrowest width that reaches the noise ceiling "
                f"({float(adequate.loc[winner, 'mean']):.3f}), using "
                f"{int(adequate.loc[winner, 'total']):,} parameters "
                f"({float(adequate.loc[winner, 'parameters_per_datum']):.2f} per target number) and "
                f"reaching {float(adequate.loc[winner, 'seed_agreement_geometry']):.3f} seed agreement "
                f"on latent drive geometry."
            )
        wider = over.drop(index=winner, errors="ignore")
        if len(wider):
            verdict += (
                f"\n\nWider still — {', '.join(f'`{i}`' for i in wider.index)} — scores up to "
                f"{float(wider['mean'].max()):.3f}, above the ceiling. That is reproducing sampling "
                f"noise, not fitting better, so width past the selected point buys overfitting "
                f"rather than accuracy. It is what makes this a constraint result rather than a "
                f"tuning one."
            )
        elif float(converged.loc[winner, "mean"]) > 1.0:
            verdict += (
                f"\n\nThe selected width sits marginally above the ceiling "
                f"({float(converged.loc[winner, 'mean']):.3f}), so it is already at the point where "
                f"extra capacity starts being spent on sampling noise."
            )
        display(Markdown(verdict))
        import yaml as _yaml
        path = TASK_ROOT / "selected_capacity.yaml"
        path.write_text(_yaml.safe_dump({
            "selected_label": str(winner),
            "hidden_units": int(str(winner).lstrip("h")),
            "selection_rule": "narrowest width reaching the noise ceiling (R2/ceiling >= 0.99), "
                              "among widths that converged on every seed; scores above the "
                              "ceiling indicate noise reproduction and are not preferred",
            "inherited_protocol": str(SELECTED_PROTOCOL["selected_label"]),
            "recurrent_bottleneck_dim": None,
            "scores": {k: float(v) for k, v in summary.loc[winner].items()},
        }, sort_keys=False))
        display(Markdown(f"Frozen to `{path}`."))

## 8. What this settles, and what comes next

**Settled.** How wide each region needs to be, measured against the noise ceiling and
with the parameter cost reported alongside — and, for the first time in this project,
without a rank constraint quietly in place.

**Two corrections carried in from setting this up**, both of which change how earlier
numbers should be read:

1. **`spectral_radius` was inert** for every run using the block parameterization, so the
   task-00 sweep's two spectral-radius arms were duplicates.
2. **The models are far smaller than reported.** `mrnn.W_rec` is a 200×200 copy the
   forward pass overwrites from the block parameters; it is registered as a parameter and
   receives no gradient. Counting it — as `model.parameters()` does — inflates the total
   about six-fold. The 50-unit model has **23,368** effective parameters against 50,400
   target numbers, a ratio of **0.46**, not the ~1 the legacy audit reported. The
   "as many parameters as data points" caveat should be retired.

**Next:** `02_connectivity.ipynb` — with width fixed, which connections are actually
necessary. Full all-to-all is the baseline; within-region only, cross-region only, and
single-pathway lesions are the constraints. Then `03_bottleneck_rank.ipynb` puts the rank
constraint back and asks how narrow inter-regional communication can be.